In [127]:
import pandas as pd 
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
import pickle
from sklearn.compose import ColumnTransformer
from sklearn.metrics  import classification_report , accuracy_score
import numpy as np


In [128]:
df = pd.read_csv('/home/aadrika.sharma/fastapi-demo-api/Serving_ML_Models_with_fastAPI/insurance.csv')

In [129]:
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
29,60,117.8,1.66,50.00,True,Lucknow,business_owner,High
54,75,54.5,1.61,3.32,True,Lucknow,retired,High
39,51,100.6,1.68,11.99,True,Bangalore,unemployed,High
62,34,72.8,1.83,35.67,False,Chennai,business_owner,Low
98,27,101.1,1.82,28.30,False,Kolkata,business_owner,Low


In [130]:
df['occupation'].unique()

array(['retired', 'freelancer', 'student', 'government_job',
       'business_owner', 'unemployed', 'private_job'], dtype=object)

In [131]:
df_feat = df.copy()

In [132]:
#Feature 1 :BMI

df_feat["bmi"] = df_feat["weight"]/(df_feat["height"]**2)


In [133]:
#Feature 2 :Age Group

def age_group(age):
    if age <18:
        return "young"
    elif age >=18 and age < 60:
        return "adult"
    else:
        return "senior"

In [134]:
df_feat["age_group"] = df_feat["age"].apply(age_group)


In [135]:
#Feature 3:Life Style Risk

def lifestyle_risk(row):
    if row["smoker"] and row["bmi"] > 30:
        return "high"
    elif row["smoker"] or row["bmi"] > 30:
        return "medium"
    else:
        return "low"

In [136]:
df_feat["lifestyle_risk"] = df_feat.apply(lifestyle_risk, axis=1)

In [137]:
tier_1_cities=["Mumbai", "Delhi", "Bangalore", "Hyderabad", "Ahmedabad", "Chennai", "Kolkata", "Pune"
               ]

tier_2_cities = ["Jaipur", "Lucknow", "Kanpur", "Nagpur", "Indore", "Thane", "Bhopal", "Visakhapatnam", "Pimpri-Chinchwad",]


In [138]:
#Feature 4:City Tier
def city_tier(city):
    if city in tier_1_cities:
        return 1
    elif city in tier_2_cities:
        return 2
    else:
        return 3

In [139]:
df_feat["city_tier"] = df_feat["city"].apply(city_tier)

In [140]:
df_feat.drop(
    columns=['age', 'weight', 'height', 'smoker', 'city']
)[
    [
        'income_lpa',
        'occupation',
        'bmi',
        'age_group',
        'lifestyle_risk',
        'insurance_premium_category'
    ]
].sample(5)

,income_lpa,occupation,bmi,age_group,lifestyle_risk,insurance_premium_category
97,44.86,freelancer,18.765432,adult,low,Low
98,28.30,business_owner,30.521676,adult,medium,Low
46,25.57,unemployed,33.672766,adult,high,High
0,2.92,retired,49.227482,senior,medium,High
70,0.57,retired,36.694215,senior,medium,High


In [141]:
print(df_feat.columns.tolist())

['age', 'weight', 'height', 'income_lpa', 'smoker', 'city', 'occupation', 'insurance_premium_category', 'bmi', 'age_group', 'lifestyle_risk', 'city_tier']


In [142]:
#Select features and target variable
X  = df_feat[["bmi", "age_group", "lifestyle_risk","city_tier", "income_lpa", "occupation"]]
Y = df_feat["insurance_premium_category"]

In [143]:
X


,bmi,age_group,lifestyle_risk,city_tier,income_lpa,occupation
0,49.227482,senior,medium,2,2.92000,retired
1,30.189017,adult,medium,1,34.28000,freelancer
2,21.118382,adult,low,2,36.64000,freelancer
3,45.535900,adult,high,1,3.34000,student
4,24.296875,senior,medium,2,3.94000,retired
...,...,...,...,...,...,...
95,21.420747,adult,low,2,19.64000,business_owner
96,47.984483,adult,medium,1,34.01000,private_job
97,18.765432,adult,low,1,44.86000,freelancer
98,30.521676,adult,medium,1,28.30000,business_owner


In [144]:
Y

0       High
1        Low
2        Low
3     Medium
4       High
       ...  
95       Low
96       Low
97       Low
98       Low
99       Low
Name: insurance_premium_category, Length: 100, dtype: object

In [145]:
#we will now split into categorial and numerical features for one hot encoding and scaling respectively

categorical_features = ["age_group", "lifestyle_risk", "occupation", "city_tier"]
numerical_features = ["bmi", "income_lpa"]

In [146]:
#now we will do one hot encoding i.e. column transformer for categorical features and passthrough for numerical features

preprocessor = ColumnTransformer(
    transformers=[
        ("car", OneHotEncoder(), categorical_features),
        ("num","passthrough", numerical_features)
    ]
)

In [147]:
#create a pipeline with preprocessing and random forest classifier 
pipeline = Pipeline9steps=[
    ("preprocessor", preprocessor),
    ("")
]

**the workflow**
                 YOUR DATASET
                      |
                      v
             Separate features and target
                      |
                      v
                 X_train, y_train
                      |
                      v
              +-------------------+
              |     PIPELINE      |
              |                   |
              |  1. PREPROCESSOR  |
              |                   |
              |  OneHotEncoder    |
              |  StandardScaler  |
              |                   |
              |         |         |
              |         v         |
              |  2. CLASSIFIER    |
              |                   |
              | Random Forest     |
              +-------------------+
                      |
                      v
                 Trained Model
                      |
                      v
              Predict new patients
                      |
                      v
          Insurance Premium Category


In [148]:
#create a pipeline with preprocessing and rando forest classifier

pipeline = Pipeline(steps = [
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state= 42))
])

In [149]:
#split the data into test and trainset
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size  = 0.2, random_state = 1)
pipeline.fit(X_train, Y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('car', OneHotEncoder(),
                                                  ['age_group',
                                                   'lifestyle_risk',
                                                   'occupation', 'city_tier']),
                                                 ('num', 'passthrough',
                                                  ['bmi', 'income_lpa'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [150]:
#predict and evaluate the model
Y_pred = pipeline.predict(X_test)
accuracy_score(Y_test, Y_pred)

0.85

In [151]:
X_test.sample(5)

,bmi,age_group,lifestyle_risk,city_tier,income_lpa,occupation
84,28.801497,senior,medium,2,0.620000,retired
52,47.344720,adult,medium,3,2.960000,student
10,22.949982,adult,medium,1,32.780000,business_owner
82,17.874812,adult,low,1,12.960000,unemployed
69,21.942857,adult,low,2,6.034487,government_job


In [155]:
import pickle

#save the trained pipeline using pickle

pickle_model_path = "model.pkl"
with open(pickle_model_path, "wb") as f:
    pickle.dump(pipeline , f)

In [153]:
import sys
import numpy
import sklearn

print("Python executable:", sys.executable)
print("NumPy version:", numpy.__version__)
print("scikit-learn version:", sklearn.__version__)

Python executable: /home/aadrika.sharma/fastapi-demo-api/myenv/bin/python
NumPy version: 1.24.4
scikit-learn version: 1.3.2
